# Out-of-Core Naïve Bayes — Theory & Implementation

A focused guide to **Out-of-Core (Incremental) Naïve Bayes** — training on datasets **too large to fit in RAM** using chunk-by-chunk learning.

> **Prerequisites:** Naïve Bayes fundamentals (Bayes' theorem, variants, Laplace smoothing) are covered in [`02-naive-bayes-from-scratch.ipynb`](02-naive-bayes-from-scratch.ipynb).

### Table of Contents
1. **Out-of-Core Learning Theory** — `partial_fit`, sufficient statistics, Welford's algorithm
2. **Data Preparation & Chunking** — simulating large-scale data loading
3. **From-Scratch Implementation** — incremental Gaussian Naïve Bayes
4. **Scikit-Learn `partial_fit`** — GaussianNB, MultinomialNB, BernoulliNB
5. **Model Evaluation** — accuracy, F1, confusion matrix
6. **Batch vs Out-of-Core Comparison** — performance & timing

> **Reference:** [sklearn Naïve Bayes docs](https://scikit-learn.org/stable/modules/naive_bayes.html) | [`GaussianNB.partial_fit`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB.partial_fit)


---
## 1. Out-of-Core Learning Theory

### 1.1 What is Out-of-Core Learning?

**Out-of-core** (also called **incremental** or **online**) learning refers to training a model on data that **cannot fit entirely in memory (RAM)** at once.

Instead of loading the full dataset, we:
1. Load a **small chunk** of data into memory
2. **Update** the model with that chunk (`partial_fit`)
3. **Discard** the chunk and load the next one
4. Repeat until all data has been seen

```
Disk: [chunk_1 | chunk_2 | chunk_3 | ... | chunk_N]
          ↓         ↓         ↓               ↓
        fit()   partial_fit partial_fit  partial_fit
          ↓         ↓         ↓               ↓
        Model ← updated incrementally ←←←←←←←←←
```

---
### 1.2 Why Naïve Bayes is Perfect for Out-of-Core Learning

Naïve Bayes is naturally suited for incremental learning because its **sufficient statistics are additive**:

| Variant | Sufficient Statistics | How to Update Incrementally |
|---------|-----------------------|-----------------------------|
| **Gaussian NB** | $n_C$, $\mu_C$, $\sigma_C^2$ per class | Welford's online algorithm for running mean & variance |
| **Multinomial NB** | Feature counts $N_{Ci}$ per class | Simply **add** new counts to existing counts |
| **Bernoulli NB** | Feature occurrence counts per class | Simply **add** new counts to existing counts |

This means you **never need to revisit old data** — just accumulate counts/statistics chunk by chunk.

---
### 1.3 The `partial_fit` API (scikit-learn)

Scikit-learn exposes `partial_fit(X, y, classes)` on Naïve Bayes estimators:

```python
clf = GaussianNB()

for chunk in data_chunks:
    X_chunk, y_chunk = chunk
    clf.partial_fit(X_chunk, y_chunk, classes=all_classes)
```

**Key rule:** Pass `classes` on the **first call** (or always) so the model knows all possible labels upfront — especially important when a chunk may not contain every class.

> **sklearn Reference:** [`GaussianNB.partial_fit`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB.partial_fit)

---
### 1.4 Welford's Online Algorithm (for Gaussian NB)

To update mean and variance **without storing all previous data**, we use Welford's method.

Given $n$ samples seen so far with running mean $\mu$ and sum of squared deviations $M_2$, when a new sample $x$ arrives:

$$n \leftarrow n + 1$$
$$\delta \leftarrow x - \mu$$
$$\mu \leftarrow \mu + \frac{\delta}{n}$$
$$\delta_2 \leftarrow x - \mu$$
$$M_2 \leftarrow M_2 + \delta \cdot \delta_2$$
$$\sigma^2 = \frac{M_2}{n}$$

This is **numerically stable** and requires $O(1)$ memory regardless of dataset size.


---
## 2. Data Preparation & Chunking


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# ── Generate a large synthetic dataset ──────────────────────────────────────
N_SAMPLES    = 100_000
N_FEATURES   = 20
N_CLASSES    = 3
CHUNK_SIZE   = 5_000      # simulate reading 5 000 rows at a time from disk
RANDOM_STATE = 42

X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=15,
    n_redundant=5,
    n_classes=N_CLASSES,
    random_state=RANDOM_STATE,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Full dataset     : {X.shape}")
print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Classes          : {np.unique(y)}")
print(f"Chunk size       : {CHUNK_SIZE}  →  {len(X_train) // CHUNK_SIZE} chunks")


In [ ]:
def get_chunks(X, y, chunk_size):
    """Yield (X_chunk, y_chunk) tuples of size chunk_size."""
    n = len(X)
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        yield X[start:end], y[start:end]

# Preview chunk sizes
chunk_sizes = [len(xc) for xc, _ in get_chunks(X_train, y_train, CHUNK_SIZE)]
print(f"Number of chunks : {len(chunk_sizes)}")
print(f"Chunk sizes      : min={min(chunk_sizes)}, max={max(chunk_sizes)}, "
      f"total={sum(chunk_sizes)}")


---
## 3. From-Scratch Out-of-Core Gaussian Naïve Bayes

We implement an **incremental Gaussian NB** that:
- Maintains **running counts, means, and M2 (sum of squared deviations)** for each class & feature
- Uses **Welford's online algorithm** for numerically stable variance updates
- Never needs to re-read previous chunks


In [ ]:
from sklearn.metrics import accuracy_score


class OutOfCoreGaussianNB:
    """
    Incremental Gaussian Naïve Bayes classifier.

    Uses Welford's online algorithm to maintain per-class, per-feature
    running mean and variance without storing any historical data.
    """

    def __init__(self, var_smoothing: float = 1e-9):
        self.var_smoothing = var_smoothing
        self._classes      = None        # sorted array of unique class labels
        self._count        = None        # shape (n_classes,)  — sample count per class
        self._mean         = None        # shape (n_classes, n_features)
        self._M2           = None        # shape (n_classes, n_features) — Welford M2
        self._total        = 0           # total samples seen

    # ── helpers ──────────────────────────────────────────────────────────────
    def _cls_idx(self, label):
        return np.searchsorted(self._classes, label)

    # ── Welford update for a single class ────────────────────────────────────
    def _welford_update(self, cls_i, X_cls):
        """Update count, mean, M2 for class index cls_i with samples X_cls."""
        for x in X_cls:
            self._count[cls_i] += 1
            n = self._count[cls_i]
            delta  = x - self._mean[cls_i]
            self._mean[cls_i] += delta / n
            delta2 = x - self._mean[cls_i]
            self._M2[cls_i]   += delta * delta2

    # ── partial_fit ──────────────────────────────────────────────────────────
    def partial_fit(self, X, y, classes=None):
        """
        Incremental fit on a chunk (X, y).

        Parameters
        ----------
        X       : array-like, shape (n_samples, n_features)
        y       : array-like, shape (n_samples,)
        classes : array-like — all possible class labels (required on first call)
        """
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        # Initialise storage on first call
        if self._classes is None:
            if classes is None:
                raise ValueError("'classes' must be provided on the first partial_fit call.")
            self._classes = np.sort(np.unique(classes))
            n_cls   = len(self._classes)
            n_feat  = X.shape[1]
            self._count = np.zeros(n_cls,          dtype=float)
            self._mean  = np.zeros((n_cls, n_feat), dtype=float)
            self._M2    = np.zeros((n_cls, n_feat), dtype=float)

        for cls_label in np.unique(y):
            ci    = self._cls_idx(cls_label)
            X_cls = X[y == cls_label]
            self._welford_update(ci, X_cls)

        self._total += len(y)
        return self

    # ── predict helpers ───────────────────────────────────────────────────────
    @property
    def _variance(self):
        var = np.zeros_like(self._M2)
        for i, n in enumerate(self._count):
            if n > 1:
                var[i] = self._M2[i] / n
        return var + self.var_smoothing

    def _log_likelihood(self, X):
        n_cls  = len(self._classes)
        log_ll = np.zeros((len(X), n_cls))
        var = self._variance
        for i in range(n_cls):
            log_ll[:, i] = -0.5 * np.sum(
                np.log(2 * np.pi * var[i])
                + ((X - self._mean[i]) ** 2) / var[i],
                axis=1
            )
        return log_ll

    def predict_log_proba(self, X):
        X         = np.asarray(X, dtype=float)
        log_prior = np.log(self._count / self._total)
        log_post  = self._log_likelihood(X) + log_prior
        log_Z     = (np.log(np.sum(np.exp(log_post - log_post.max(axis=1, keepdims=True)),
                                    axis=1, keepdims=True))
                     + log_post.max(axis=1, keepdims=True))
        return log_post - log_Z

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        return self._classes[np.argmax(self.predict_log_proba(X), axis=1)]

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))


print("OutOfCoreGaussianNB class defined.")


In [ ]:
import time

ALL_CLASSES = np.unique(y_train)

scratch_clf = OutOfCoreGaussianNB()

start = time.perf_counter()
for X_chunk, y_chunk in get_chunks(X_train, y_train, CHUNK_SIZE):
    scratch_clf.partial_fit(X_chunk, y_chunk, classes=ALL_CLASSES)
scratch_time = time.perf_counter() - start

scratch_acc = scratch_clf.score(X_test, y_test)
print(f"[From Scratch] Training time : {scratch_time:.4f} s")
print(f"[From Scratch] Test accuracy : {scratch_acc:.4f}")


---
## 4. Out-of-Core Naïve Bayes with Scikit-Learn (`partial_fit`)

Scikit-learn provides native incremental support through `partial_fit()` on:

| Class | Feature Type | Use Case |
|-------|-------------|----------|
| [`GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html) | Continuous (real-valued) | General classification |
| [`MultinomialNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) | Non-negative integer counts | Text / NLP |
| [`BernoulliNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.BernoulliNB.html) | Binary (0/1) features | Binary feature sets |

> **sklearn docs:** [Naïve Bayes — User Guide](https://scikit-learn.org/stable/modules/naive_bayes.html)


In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()

start = time.perf_counter()
for X_chunk, y_chunk in get_chunks(X_train, y_train, CHUNK_SIZE):
    gnb.partial_fit(X_chunk, y_chunk, classes=ALL_CLASSES)
gnb_time = time.perf_counter() - start

gnb_acc = gnb.score(X_test, y_test)
print(f"[GaussianNB partial_fit] Training time : {gnb_time:.4f} s")
print(f"[GaussianNB partial_fit] Test accuracy : {gnb_acc:.4f}")


In [ ]:
from sklearn.naive_bayes import BernoulliNB

X_bin_train = (X_train > np.median(X_train, axis=0)).astype(int)
X_bin_test  = (X_test  > np.median(X_train, axis=0)).astype(int)

bnb = BernoulliNB()

start = time.perf_counter()
for X_chunk, y_chunk in get_chunks(X_bin_train, y_train, CHUNK_SIZE):
    bnb.partial_fit(X_chunk, y_chunk, classes=ALL_CLASSES)
bnb_time = time.perf_counter() - start

bnb_acc = bnb.score(X_bin_test, y_test)
print(f"[BernoulliNB partial_fit] Training time : {bnb_time:.4f} s")
print(f"[BernoulliNB partial_fit] Test accuracy : {bnb_acc:.4f}")


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler

scaler      = MinMaxScaler()
X_cnt_train = (scaler.fit_transform(X_train) * 100).astype(int)
X_cnt_test  = (scaler.transform(X_test)      * 100).astype(int)

mnb = MultinomialNB()

start = time.perf_counter()
for X_chunk, y_chunk in get_chunks(X_cnt_train, y_train, CHUNK_SIZE):
    mnb.partial_fit(X_chunk, y_chunk, classes=ALL_CLASSES)
mnb_time = time.perf_counter() - start

mnb_acc = mnb.score(X_cnt_test, y_test)
print(f"[MultinomialNB partial_fit] Training time : {mnb_time:.4f} s")
print(f"[MultinomialNB partial_fit] Test accuracy : {mnb_acc:.4f}")


---
## 5. Model Evaluation


In [ ]:
from sklearn.metrics import classification_report

y_pred_gnb     = gnb.predict(X_test)
y_pred_scratch = scratch_clf.predict(X_test)

print("=" * 55)
print("  sklearn GaussianNB (partial_fit)")
print("=" * 55)
print(classification_report(y_test, y_pred_gnb,
                             target_names=[f"Class {c}" for c in ALL_CLASSES]))

print("=" * 55)
print("  From-Scratch OutOfCoreGaussianNB")
print("=" * 55)
print(classification_report(y_test, y_pred_scratch,
                             target_names=[f"Class {c}" for c in ALL_CLASSES]))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(
    axes,
    [y_pred_gnb, y_pred_scratch],
    ["sklearn GaussianNB\n(partial_fit)", "From-Scratch\nOutOfCoreGaussianNB"],
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=[f"C{c}" for c in ALL_CLASSES],
                yticklabels=[f"C{c}" for c in ALL_CLASSES])
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices — Out-of-Core Gaussian Naïve Bayes", fontsize=13)
plt.tight_layout()
plt.show()


---
## 6. Batch vs Out-of-Core Comparison

We compare three setups:
1. **Batch GaussianNB** — `fit()` on all training data at once
2. **sklearn GaussianNB** — `partial_fit()` chunk by chunk
3. **From-Scratch** — custom incremental Gaussian NB


In [ ]:
import tracemalloc

def measure(train_fn):
    """Return (accuracy, elapsed_sec, peak_memory_kb)."""
    tracemalloc.start()
    t0 = time.perf_counter()
    clf = train_fn()
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    acc = clf.score(X_test, y_test)
    return acc, elapsed, peak / 1024  # peak in KB


# 1. Batch fit
def train_batch():
    clf = GaussianNB()
    clf.fit(X_train, y_train)
    return clf

# 2. sklearn partial_fit
def train_sklearn_partial():
    clf = GaussianNB()
    for X_c, y_c in get_chunks(X_train, y_train, CHUNK_SIZE):
        clf.partial_fit(X_c, y_c, classes=ALL_CLASSES)
    return clf

# 3. From-scratch partial fit
def train_scratch_partial():
    clf = OutOfCoreGaussianNB()
    for X_c, y_c in get_chunks(X_train, y_train, CHUNK_SIZE):
        clf.partial_fit(X_c, y_c, classes=ALL_CLASSES)
    return clf


results = {}
results["Batch\nGaussianNB"]            = measure(train_batch)
results["sklearn\npartial_fit"]          = measure(train_sklearn_partial)
results["From-Scratch\npartial_fit"]     = measure(train_scratch_partial)

print(f"{'Method':<28} {'Accuracy':>10} {'Time (s)':>10} {'Peak Mem (KB)':>15}")
print("-" * 65)
for method, (acc, sec, mem) in results.items():
    label = method.replace('\n', ' ')
    print(f"{label:<28} {acc:>10.4f} {sec:>10.4f} {mem:>15.1f}")


In [ ]:
# ── Comparison bar charts ────────────────────────────────────────────────────
labels  = list(results.keys())
accs    = [v[0] for v in results.values()]
times   = [v[1] for v in results.values()]
mems    = [v[2] for v in results.values()]
colors  = ["#4C72B0", "#55A868", "#C44E52"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Batch vs Out-of-Core Naïve Bayes — Comparison", fontsize=14, fontweight="bold")

# Accuracy
axes[0].bar(labels, accs, color=colors, edgecolor="black", alpha=0.85)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Test Accuracy")
axes[0].set_ylim(max(0, min(accs) - 0.05), min(1.0, max(accs) + 0.05))
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.001, f"{v:.4f}", ha="center", va="bottom", fontsize=10)

# Training Time
axes[1].bar(labels, times, color=colors, edgecolor="black", alpha=0.85)
axes[1].set_ylabel("Time (seconds)")
axes[1].set_title("Training Time")
for i, v in enumerate(times):
    axes[1].text(i, v + 0.0005, f"{v:.3f}s", ha="center", va="bottom", fontsize=10)

# Peak Memory
axes[2].bar(labels, mems, color=colors, edgecolor="black", alpha=0.85)
axes[2].set_ylabel("Peak Memory (KB)")
axes[2].set_title("Peak Memory Usage")
for i, v in enumerate(mems):
    axes[2].text(i, v + 5, f"{v:.0f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# ── Accuracy evolution over chunks (learning curve) ──────────────────────────
clf_live = GaussianNB()
chunk_accs = []

for X_chunk, y_chunk in get_chunks(X_train, y_train, CHUNK_SIZE):
    clf_live.partial_fit(X_chunk, y_chunk, classes=ALL_CLASSES)
    chunk_accs.append(clf_live.score(X_test, y_test))

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(chunk_accs) + 1), chunk_accs, marker="o",
         color="#4C72B0", linewidth=2, markersize=5)
plt.axhline(y=gnb.score(X_test, y_test), linestyle="--", color="gray",
            label="Final accuracy (all chunks)")
plt.xlabel("Chunks processed")
plt.ylabel("Test Accuracy")
plt.title("sklearn GaussianNB — Accuracy After Each Chunk (Out-of-Core Learning Curve)")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


---
## Summary & Key Takeaways

| Topic | Key Point |
|-------|-----------|
| **Out-of-Core** | Trains on data too large for RAM by processing chunks one at a time |
| **`partial_fit`** | sklearn's incremental API — always pass `classes=` on first call |
| **Sufficient Statistics** | Naïve Bayes only needs counts, means, variances — all additive |
| **Welford's Algorithm** | Numerically stable online mean/variance update in $O(1)$ memory |
| **Accuracy** | Batch and incremental GaussianNB converge to the same accuracy |
| **Memory** | Out-of-core uses far less peak memory vs loading full dataset |
| **Chunk Size** | Larger chunks = fewer updates, faster training; smaller = lower memory |

### sklearn Reference Links
- [Naïve Bayes — User Guide](https://scikit-learn.org/stable/modules/naive_bayes.html)
- [`GaussianNB.partial_fit`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB.partial_fit)
- [`MultinomialNB.partial_fit`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB.partial_fit)
- [`BernoulliNB.partial_fit`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.BernoulliNB.html#sklearn.naive_bayes.BernoulliNB.partial_fit)
- [Out-of-Core Classification Example](https://scikit-learn.org/stable/auto_examples/applications/plot_out_of_core_classification.html)
